<a href="https://colab.research.google.com/github/ealuede123-alt/DTSC-3020/blob/main/AluedeProject3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project 3 — Eagle Beans Data Day ☕📊

**Course:** DTSC 3020
**Covers:** Lessons 5–6 — functions
**Points:** 100 (5 parts)  
**Released:** Monday, July 6, 2026  
**Due:** Monday, July 13, 2026 at 11:59 PM

---

## Before you start — rename this file

Make your own copy using **File → Save a copy in Drive**. Rename it so your last name appears first, for example:

`SmithMiniProject3.ipynb`

An incorrectly named file may lose submission-format points.

---

## The story

Eagle Beans Coffee has been recording its sales, and the owner wants a complete **end-of-day data pipeline**. In Mini Project 2, you built the register logic. In this project, you will turn one day of raw sales into a clean report using tools from Lessons 5 and 6:

1. **Part 1** — Write reusable **functions** for the shop's calculations *(20 pts)*
2. **Part 2** — Save the day's sales to a **CSV file** and read it back *(20 pts)*
3. **Part 3** — Analyze the numbers with **NumPy** *(20 pts)*
4. **Part 4** — Analyze the day with a **pandas DataFrame** *(20 pts)*
5. **Part 5** — Build a dated report with **`datetime` + regex** and save it to a file *(20 pts)*

**This is one connected program.** Part 2 uses the receipt function from Part 1 and creates the CSV file used in Parts 3–5. Run the notebook **in order from top to bottom**.

> 💡 The project is self-contained. Every required file is created by your own code, so **Runtime → Restart and run all** should reproduce all results without manual uploads.

---

## Rules

- Start **every** answer cell with a comment containing your name and the part number.
- Use meaningful function and variable names, such as `make_receipt_total` instead of `f1`.
- Do not hard-code expected results. Totals, statistics, filters, counts, and report values must be calculated from the provided data.
- Equivalent correct solutions are accepted unless a part explicitly requires a specific function or method.
- Before submitting, use **Runtime → Restart and run all** and confirm that every cell runs without errors in order.
- After the notebook finishes running, **save it again** so the outputs are visible in the GitHub version.

## Grading (100 points)

| Part | Skills | Points |
|---|---|---:|
| 1 | Defining functions: parameters, `return`, default argument, and function calls | 20 |
| 2 | `os`, writing and reading a CSV file, type conversion, and a running total | 20 |
| 3 | NumPy array, statistics, Boolean indexing, and vectorized math | 20 |
| 4 | pandas DataFrame: load, create a column, summarize, filter, and group | 20 |
| 5 | `datetime`, `re` (regex), and writing and reading a summary text file | 20 |

Clean code, clear comments, meaningful names, and successful top-to-bottom execution are part of each section's score.

---

## How to submit

Push this completed notebook to your GitHub account and paste the GitHub URL in Canvas.

In Colab, select **File → Save a copy in GitHub**. Open the notebook on GitHub, confirm that the code and outputs are visible, copy the full URL from your browser, and paste it into Canvas.

Submit a GitHub URL to your completed **`.ipynb` notebook**. Do not submit a `.py` file, PDF, screenshot, or local file path.


## Setup (run this first)

In [1]:
# Run this cell first — it imports everything the project needs.
import os
import csv
import re
import datetime
import numpy as np
import pandas as pd

print("☕ Eagle Beans Data Day — libraries loaded, let's build the pipeline!")

☕ Eagle Beans Data Day — libraries loaded, let's build the pipeline!


## Part 1 — The Shop's Function Toolkit *(20 points)*

Before processing the sales data, create two reusable functions for the shop's calculations.

### What to do

1. Start the answer cell with two comment lines: your name and the part number.
2. Define a function `make_receipt_total(price, quantity, tax_rate=0.0825)` that:
   - calculates `subtotal = price * quantity`,
   - adds tax using `subtotal * tax_rate`,
   - **returns** the final total rounded to 2 decimal places.

   The default `tax_rate` is `0.0825`, but the caller must be able to override it.

3. Define a second function `stars_for(total)` that **returns** the number of loyalty stars earned: **1 star for each complete $5** in the total.

   Hint:

   ```python
   int(total) // 5
   ```

4. Test the functions by printing labeled results for all three calls:

   ```python
   make_receipt_total(4.50, 3)        # expected: 14.61
   make_receipt_total(3.25, 2, 0.0)   # expected: 6.5 or 6.50
   stars_for(14.61)                    # expected: 2
   ```

5. Add a comment explaining the difference between a function **parameter** and an **argument**.

### Hints

- Use `def function_name(parameters):` and `return value`.
- `make_receipt_total()` will be used again in Part 2, so it must return a value instead of only printing one.


In [2]:
# Lam Bob
# Part 1 — The shop's function toolkit

# A parameter is the variable name in the function definition (e.g., price, quantity);
# an argument is the actual value supplied by the caller when the function is called
# (e.g., 4.50, 3).

def make_receipt_total(price, quantity, tax_rate=0.0825):
    """Return the tax-included total for an order, rounded to 2 decimal places."""
    subtotal = price * quantity
    total = subtotal + subtotal * tax_rate
    return round(total, 2)

def stars_for(total):
    """Return loyalty stars earned: 1 star per complete $5 in the total."""
    return int(total) // 5

print(f"make_receipt_total(4.50, 3)       -> {make_receipt_total(4.50, 3)}")
print(f"make_receipt_total(3.25, 2, 0.0)  -> {make_receipt_total(3.25, 2, 0.0)}")
print(f"stars_for(14.61)                  -> {stars_for(14.61)}")


make_receipt_total(4.50, 3)       -> 14.61
make_receipt_total(3.25, 2, 0.0)  -> 6.5
stars_for(14.61)                  -> 2


## Part 2 — Save and Reload the Day's Sales (CSV) *(20 points)*

Save the day's order data to a CSV file, read it back, and calculate each tax-included receipt total using the function from Part 1.

**Run Part 1 first** because this section calls `make_receipt_total()`.

### Provided data

```python
sales_rows = [
    ["customer", "item", "price", "qty"],
    ["maria", "latte", "4.50", "3"],
    ["devon", "mocha", "5.00", "1"],
    ["aisha", "espresso", "3.50", "2"],
    ["carlos", "latte", "4.50", "1"],
    ["lena", "muffin", "3.25", "4"],
]
```

### What to do

1. Use `csv.writer` to write `sales_rows` to a file named `eagle_sales.csv`. Open the file with `newline=""` and `encoding="utf-8"`.
2. Use the `os` module to confirm that `eagle_sales.csv` exists, and print the result.
3. Open `eagle_sales.csv` in read mode with `encoding="utf-8"` and use `csv.reader`.
4. Skip the header using `next(reader)`.
5. Loop through the remaining rows. For each row:
   - convert `price` to `float`,
   - convert `qty` to `int`,
   - call `make_receipt_total(price, qty)`,
   - print a labeled line such as `maria: $14.61`,
   - add the **rounded receipt total returned by the function** to `grand_total`.
6. Print the tax-included `grand_total` formatted as money.

   Expected grand total:

   ```text
   Grand total: $46.54
   ```

7. Add a comment explaining why `price` and `qty` must be converted before performing calculations.

### Hints

- `csv.writer(file).writerows(sales_rows)` writes all rows.
- Use `with open("eagle_sales.csv", "w", newline="", encoding="utf-8")` when writing.
- Values read with `csv.reader` are strings, so use conversions such as `float("4.50")` and `int("3")`.
- Add each value returned by `make_receipt_total()` to the running total. Do not recalculate one combined tax amount at the end.


In [3]:
# Lam Bob
# Part 2 — Save and reload the day's sales

# CSV reader returns every value as a string; float() and int() are needed
# to convert them into numbers before any arithmetic can be performed.

sales_rows = [
    ["customer", "item", "price", "qty"],
    ["maria",  "latte",    "4.50", "3"],
    ["devon",  "mocha",    "5.00", "1"],
    ["aisha",  "espresso", "3.50", "2"],
    ["carlos", "latte",    "4.50", "1"],
    ["lena",   "muffin",   "3.25", "4"],
]

with open("eagle_sales.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(sales_rows)

print(f"eagle_sales.csv exists? {os.path.exists('eagle_sales.csv')}")

grand_total = 0.0
with open("eagle_sales.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        customer, item, price_str, qty_str = row
        price = float(price_str)
        qty   = int(qty_str)
        receipt = make_receipt_total(price, qty)
        grand_total += receipt
        print(f"{customer}: ${receipt:.2f}")

print(f"Grand total: ${grand_total:.2f}")


eagle_sales.csv exists? True
maria: $14.61
devon: $5.41
aisha: $7.58
carlos: $4.87
lena: $14.07
Grand total: $46.54


## Part 3 — Number Crunching with NumPy *(20 points)*

Convert the **pre-tax line totals** (`price × qty`) into a NumPy array and analyze them.

**Run Part 2 first** so `eagle_sales.csv` exists.

### What to do

1. Read `eagle_sales.csv` again and skip the header.
2. For each data row, calculate the **pre-tax** line total `price * qty` and store it in a Python list named `line_totals`.

   > These values are pre-tax, so they are lower than the receipt totals from Part 2.

3. Convert `line_totals` to a NumPy array named `totals`, then print the array.
4. Print each of the following with a clear label:
   - mean, rounded to 2 decimal places,
   - maximum,
   - minimum,
   - population standard deviation using `np.std`, rounded to 2 decimal places.

   Expected numerical values:

   ```text
   Mean: 8.60
   Maximum: 13.5
   Minimum: 4.5
   Population standard deviation: 3.89
   ```

   Equivalent formatting such as `8.6` instead of `8.60` is acceptable.
5. Use Boolean indexing to print the values above the mean:

   ```python
   totals[totals > totals.mean()]
   ```

6. Apply a **10% increase** to every value using one vectorized operation and no loop:

   ```python
   totals * 1.10
   ```

7. Add a comment naming one advantage of a NumPy array over a regular Python list for this calculation.

### Hints

- Build the Python list first, then use `totals = np.array(line_totals)`.
- `totals.mean()` and `np.mean(totals)` are equivalent.
- Equivalent numeric formatting is acceptable when the value is correct.


In [4]:
# Lam Bob
# Part 3 — Number crunching with NumPy

# A NumPy array supports vectorized operations (e.g., totals * 1.10) that apply
# to every element at once without a Python loop, making bulk calculations much
# faster and more concise than a plain Python list.

line_totals = []
with open("eagle_sales.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        _, _, price_str, qty_str = row
        line_totals.append(float(price_str) * int(qty_str))

totals = np.array(line_totals)
print(f"Pre-tax line totals: {totals}")
print(f"Mean:                          {round(totals.mean(), 2)}")
print(f"Maximum:                       {totals.max()}")
print(f"Minimum:                       {totals.min()}")
print(f"Population standard deviation: {round(np.std(totals), 2)}")
print(f"Values above mean:             {totals[totals > totals.mean()]}")
print(f"With 10% increase:             {np.round(totals * 1.10, 2)}")


Pre-tax line totals: [13.5  5.   7.   4.5 13. ]
Mean:                          8.6
Maximum:                       13.5
Minimum:                       4.5
Population standard deviation: 3.89
Values above mean:             [13.5 13. ]
With 10% increase:             [14.85  5.5   7.7   4.95 14.3 ]


## Part 4 — The Day in a pandas DataFrame *(20 points)*

Use pandas to load and analyze the same CSV file.

**Run Part 2 first** so `eagle_sales.csv` exists.

### What to do

1. Load the file using:

   ```python
   df = pd.read_csv("eagle_sales.csv")
   ```

   Print the DataFrame.

   > For this file, pandas will infer `price` as numeric values and `qty` as integers, so no manual conversion is needed.

2. Create a new column named `line_total`:

   ```python
   df["line_total"] = df["price"] * df["qty"]
   ```

   Print the updated DataFrame.

3. Print each of the following with a clear label:
   - DataFrame shape using `df.shape`,
   - total pre-tax revenue using the sum of `line_total`, rounded to 2 decimal places,
   - average `line_total`, rounded to 2 decimal places.

4. Filter the DataFrame and print only the `customer` and `line_total` columns for rows where `line_total` is greater than 10.
5. Group by `item` and print total revenue per item:

   ```python
   df.groupby("item")["line_total"].sum()
   ```

6. Print the **most frequently listed item by number of order rows**:

   ```python
   df["item"].value_counts().idxmax()
   ```

   This counts how many rows contain each item; it does not add the `qty` values.

7. Add a comment explaining, in one sentence, what `groupby` allows you to do that a single column sum does not.

### Hints

- Create a column by assigning values to `df["new_column"]`.
- `df[df["line_total"] > 10]` keeps matching rows.
- Add `[["customer", "line_total"]]` to select only those two columns.
- `value_counts()` counts occurrences, and `.idxmax()` returns the label with the largest count.


In [5]:
# Lam Bob
# Part 4 — The day in a pandas DataFrame

# groupby lets you compute a statistic (e.g., sum) separately for each category
# in a column, whereas a single column sum collapses everything into one number
# and loses the per-group breakdown.

df = pd.read_csv("eagle_sales.csv")
print("Loaded DataFrame:")
print(df)
print()

df["line_total"] = df["price"] * df["qty"]
print("With line_total column:")
print(df)
print()

print(f"Shape:               {df.shape}")
print(f"Total pre-tax revenue: ${round(df['line_total'].sum(), 2):.2f}")
print(f"Average line_total:    ${round(df['line_total'].mean(), 2):.2f}")
print()

high_spenders = df[df["line_total"] > 10][["customer", "line_total"]]
print("Orders with line_total > $10:")
print(high_spenders)
print()

print("Revenue per item:")
print(df.groupby("item")["line_total"].sum())
print()

top_item = df["item"].value_counts().idxmax()
print(f"Most frequently ordered item: {top_item}")


Loaded DataFrame:
  customer      item  price  qty
0    maria     latte   4.50    3
1    devon     mocha   5.00    1
2    aisha  espresso   3.50    2
3   carlos     latte   4.50    1
4     lena    muffin   3.25    4

With line_total column:
  customer      item  price  qty  line_total
0    maria     latte   4.50    3        13.5
1    devon     mocha   5.00    1         5.0
2    aisha  espresso   3.50    2         7.0
3   carlos     latte   4.50    1         4.5
4     lena    muffin   3.25    4        13.0

Shape:               (5, 5)
Total pre-tax revenue: $43.00
Average line_total:    $8.60

Orders with line_total > $10:
  customer  line_total
0    maria        13.5
4     lena        13.0

Revenue per item:
item
espresso     7.0
latte       18.0
mocha        5.0
muffin      13.0
Name: line_total, dtype: float64

Most frequently ordered item: latte


## Part 5 — The Dated Report (datetime + regex + file write) *(20 points)*

Finish the pipeline by creating a dated text report, validating customer email addresses with a regular expression, and saving the report to a file.

**Run Parts 2 and 4 first** because this section uses `eagle_sales.csv` and `df`.

### What to do

1. Create the fixed report date below so every student's output is reproducible:

   ```python
   report_date = datetime.date(2026, 7, 13)
   ```

2. Create a formatted date string and print it:

   ```python
   formatted_date = report_date.strftime("%A, %B %d, %Y")
   ```

   Expected output:

   ```text
   Monday, July 13, 2026
   ```

3. Use the following email list:

   ```python
   emails = [
       "maria@unt.edu",
       "devon@@bad",
       "aisha@gmail.com",
       "carlos.nodomain",
       "lena@my.unt.edu",
   ]
   ```

4. Use `re.match()` with this raw-string pattern to build and print a list named `valid_emails`:

   ```python
   r"^[\w.]+@[\w.]+\.\w+$"
   ```

5. Create a text report and write it to `daily_summary.txt` using `with open(..., "w", encoding="utf-8")`.

   The report must contain these three lines:

   ```text
   Eagle Beans Daily Summary - Monday, July 13, 2026
   Total revenue: $43.00
   Valid customer emails: 3
   ```

   Calculate the revenue and valid-email count in code. Do not hard-code `$43.00` or `3`.

6. Use the `os` module to confirm that `daily_summary.txt` exists and print the result.
7. Open the file in read mode with `encoding="utf-8"` and print its full contents.
8. Add a comment explaining why a fixed `report_date` is used instead of `datetime.date.today()`.

### Hints

- Use `formatted_date` in the report title.
- `file.write(text)` writes one string; include `\n` between lines.
- Use the regular keyboard hyphen (`-`) in the report title, exactly as shown.
- Revenue can be calculated with `df["line_total"].sum()`.
- Reuse the email-validation idea from Assignment 5, Problem 4.


In [6]:
# Lam Bob
# Part 5 — The dated report

# A fixed report_date is used instead of datetime.date.today() so that the output
# is the same every time the notebook is graded, regardless of when it is run.

report_date   = datetime.date(2026, 7, 13)
formatted_date = report_date.strftime("%A, %B %d, %Y")
print(f"Report date: {formatted_date}")

emails = [
    "maria@unt.edu",
    "devon@@bad",
    "aisha@gmail.com",
    "carlos.nodomain",
    "lena@my.unt.edu",
]

valid_emails = [e for e in emails if re.match(r"^[\w.]+@[\w.]+\.\w+$", e)]
print(f"Valid emails: {valid_emails}")

total_revenue = df["line_total"].sum()

report_lines = [
    f"Eagle Beans Daily Summary - {formatted_date}\n",
    f"Total revenue: ${total_revenue:.2f}\n",
    f"Valid customer emails: {len(valid_emails)}\n",
]

with open("daily_summary.txt", "w", encoding="utf-8") as f:
    f.writelines(report_lines)

print(f"daily_summary.txt exists? {os.path.exists('daily_summary.txt')}")

with open("daily_summary.txt", "r", encoding="utf-8") as f:
    print(f.read())


Report date: Monday, July 13, 2026
Valid emails: ['maria@unt.edu', 'aisha@gmail.com', 'lena@my.unt.edu']
daily_summary.txt exists? True
Eagle Beans Daily Summary - Monday, July 13, 2026
Total revenue: $43.00
Valid customer emails: 3



## Final check before submitting ✅

1. Select **Runtime → Restart and run all**. Every cell must run without errors from top to bottom.
2. Confirm that each answer cell begins with your **name and part number** in comments.
3. Confirm that both Part 1 functions use `return`, and that `make_receipt_total()` is called in Part 2.
4. Confirm that your results are calculated from the provided data rather than hard-coded.
5. Confirm that `eagle_sales.csv` and `daily_summary.txt` are created by your code without manual uploads.
6. Confirm that the report date is **Monday, July 13, 2026** and the title uses the regular hyphen shown in the required output.
7. Save the notebook after running it so all outputs remain visible.
8. Confirm that the file name begins with your last name, for example `SmithMiniProject3.ipynb`.
9. Use **File → Save a copy in GitHub**, open the notebook on GitHub, and verify that the code and outputs are visible.
10. Paste the GitHub URL to the `.ipynb` notebook into Canvas.

Nice work — you turned a day of raw sales into a clean, dated report. That is a complete data pipeline! ☕📊
